# Lab 5 — JetAuto Robot: Inspection, Connection, and Movement

<svg width="100%" viewBox="0 0 1260 150" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Physical robot workflow">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#334155"/></marker></defs>
<rect x="0" y="0" width="1260" height="150" rx="18" fill="#f8fafc" stroke="#cbd5e1"/>
<text x="24" y="30" font-family="Arial" font-size="20" font-weight="700" fill="#0f172a">Physical robot workflow</text>
<rect x="25.0" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="115.4" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Inspect hardware</text>
<line x1="205.8" y1="84" x2="224.8" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="230.8" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="321.2" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Connect by Wi‑Fi or</text>
<text x="321.2" y="97" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">USB</text>
<line x1="411.7" y1="84" x2="430.7" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="436.7" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="527.1" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">SSH / remote</text>
<text x="527.1" y="97" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">desktop</text>
<line x1="617.5" y1="84" x2="636.5" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="642.5" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="732.9" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Stop app service</text>
<line x1="823.3" y1="84" x2="842.3" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="848.3" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="938.8" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Launch controller</text>
<line x1="1029.2" y1="84" x2="1048.2" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="1054.2" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="1144.6" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Test motion safely</text>
</svg>


## Learning outcomes
By the end of this lab, you will be able to:

1. Inspect the JetAuto robot before powering or moving it.
2. Connect to the robot using Wi‑Fi + SSH, with USB serial as a fallback.
3. Stop conflicting startup services before taking ROS control.
4. Verify the active ROS graph, command topic, and controller status.
5. Send safe movement commands to the physical robot.
6. Transfer and run your Lab 4 controller package on the real JetAuto.
7. Compare simulation behavior with real-world robot behavior.

## Lab environment

| Item | Main lab target |
|---|---|
| Robot | JetAuto physical robot |
| OS / ROS | Ubuntu 18.04 + ROS1 Melodic on JetAuto / Jetson Nano |
| Student computer | Ubuntu VM or Linux laptop connected to JetAuto Wi‑Fi |
| Main control topic | `/jetauto_controller/cmd_vel` |
| Message type | `geometry_msgs/Twist` |

> **Important:** Run ROS commands in a Linux terminal unless a cell explicitly says it is a Python helper or visualization cell.


## Safety-first checklist
Before any motion test, confirm the following:

- Battery charger is unplugged.
- Robot is on the ground, not on a desk or table.
- Wheels have open space around them.
- Loose cables, nuts, bolts, and accessories are secured.
- LiDAR/camera brackets are not loose.
- You know the stop command and can paste/run it quickly.
- Linear speed stays within the instructor-approved safe range.
- Only one person is operating the robot at a time.

### Emergency stop command
Keep this command ready in a terminal:

```bash
rostopic pub -1 /jetauto_controller/cmd_vel geometry_msgs/Twist \
'{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}'
```


## Part 1 — Inspect the robot
Validate the robot using the kit/manual guidance:

- Battery and power switch condition
- Visible cable connections
- Wheel modules and structural screws
- LiDAR/camera mounting
- No missing or loose parts
- No objects trapped around the mecanum wheels

Do **not** use the phone app for this lab workflow. ROS will control the robot directly.

### Verification checkpoint
Record one or two photos of the physical setup showing that the robot is safely positioned before motion testing.


## Part 2 — Connect by Wi‑Fi and SSH
The JetAuto runs as a Wi‑Fi access point. Connect your computer or VM to the robot SSID, then SSH into the robot:

```bash
ssh jetauto@192.168.149.1
```

Default course credentials are typically provided by the instructor. If they differ, use your lab-specific credentials.

> **Note:** The robot access point usually has no internet. Save these instructions locally before disconnecting from your normal network.

### Verify the connection
On the robot, run:

```bash
hostname
ip addr
rosversion -d
```

Expected ROS distribution for the main lab:

```bash
melodic
```


## Optional fallback — USB serial connection
If Wi‑Fi is unreliable, use a USB serial session:

```bash
sudo apt-get update
sudo apt-get install -y screen
sudo screen /dev/ttyACM0 115200
```

The device name may differ. Check recent kernel messages after plugging in the cable:

```bash
dmesg | tail -30
ls /dev/ttyACM* /dev/ttyUSB* 2>/dev/null
```

To exit `screen`, press:

```text
Ctrl+A, then K, then Y
```


## Optional GUI access — NoMachine
NoMachine can provide GUI access to the robot desktop. Use it when you need graphical tools such as RViz. Use SSH for lighter command-line control.

Recommended workflow:

- Use SSH for controller launch, topic checks, and motion commands.
- Use NoMachine only when a GUI is required.
- Avoid running multiple controllers from multiple sessions at the same time.


## Part 3 — Start hardware control
On the robot, stop the default startup app service before launching the ROS controller:

```bash
sudo systemctl stop start_app_node.service
roslaunch jetauto_controller jetauto_controller.launch
```

Stopping the app service prevents the mobile app controller from competing with your ROS commands.

### Verify active ROS topics
Open another SSH terminal and run:

```bash
rostopic list
```

Look for the command velocity topic:

```bash
/jetauto_controller/cmd_vel
```

Check its message type:

```bash
rostopic type /jetauto_controller/cmd_vel
```

Expected output:

```text
geometry_msgs/Twist
```


## Part 4 — Safe motion commands
Always begin and end with a stop command.

### 4.1 Stop command

```bash
rostopic pub -1 /jetauto_controller/cmd_vel geometry_msgs/Twist \
'{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}'
```

### 4.2 Small forward test

```bash
rostopic pub -1 /jetauto_controller/cmd_vel geometry_msgs/Twist \
'{linear: {x: 0.1, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}'
```

Immediately send the stop command again.

### 4.3 Small rotation test

```bash
rostopic pub -1 /jetauto_controller/cmd_vel geometry_msgs/Twist \
'{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.2}}'
```

Immediately send the stop command again.

### 4.4 Observe the command topic
In a separate terminal:

```bash
rostopic echo /jetauto_controller/cmd_vel
```

Then publish one of the movement commands above and confirm that the message appears.


## Part 5 — Use your Lab 4 controller package on the real robot
In Lab 4, you created or used a controller package for simulated motion. In this lab, you will run the same idea on the physical JetAuto robot.

### 5.1 Transfer your package to the robot
From your student computer, copy your package or workspace to the robot:

```bash
scp -r ~/catkin_ws/src/lab4_jetauto_control jetauto@192.168.149.1:~/catkin_ws/src/
```

On the robot:

```bash
cd ~/catkin_ws
catkin_make
source devel/setup.bash
```

### 5.2 Update the topic name if needed
Your Lab 4 controller may publish to `/cmd_vel` for simulation. The physical JetAuto controller uses:

```text
/jetauto_controller/cmd_vel
```

Update your Python code so the publisher uses the physical robot topic:

```python
pub = rospy.Publisher('/jetauto_controller/cmd_vel', Twist, queue_size=10)
```

### 5.3 Run your controller
Example:

```bash
rosrun lab4_jetauto_control teleop_key_control.py
```

Send the stop command after testing.


## Optional ROS1 keyboard controller package
If you did not complete a reusable controller in Lab 4, you may create a simple keyboard controller package for this lab.

### 1. Create the package

```bash
cd ~/catkin_ws/src
catkin_create_pkg lab5_jetauto_control rospy geometry_msgs
mkdir -p lab5_jetauto_control/scripts
```

### 2. Create the controller script

```bash
nano ~/catkin_ws/src/lab5_jetauto_control/scripts/teleop_key_control.py
```

Paste the following code:

```python
#!/usr/bin/env python3

import sys
import termios
import tty

import rospy
from geometry_msgs.msg import Twist


def get_key():
    """Read one keyboard key without requiring Enter."""
    settings = termios.tcgetattr(sys.stdin)
    try:
        tty.setraw(sys.stdin.fileno())
        key = sys.stdin.read(1)
    finally:
        termios.tcsetattr(sys.stdin, termios.TCSADRAIN, settings)
    return key


def make_twist(x=0.0, y=0.0, z_rot=0.0):
    """Create a Twist message for JetAuto base motion."""
    msg = Twist()
    msg.linear.x = x
    msg.linear.y = y
    msg.angular.z = z_rot
    return msg


def main():
    rospy.init_node('lab5_jetauto_keyboard_control')
    pub = rospy.Publisher('/jetauto_controller/cmd_vel', Twist, queue_size=10)

    speed = 0.10
    turn = 0.25

    print('JetAuto keyboard control')
    print('w: forward | s: backward | a: strafe left | d: strafe right')
    print('q: rotate left | e: rotate right | space: stop | x: exit')

    while not rospy.is_shutdown():
        key = get_key()

        if key == 'w':
            cmd = make_twist(x=speed)
        elif key == 's':
            cmd = make_twist(x=-speed)
        elif key == 'a':
            cmd = make_twist(y=speed)
        elif key == 'd':
            cmd = make_twist(y=-speed)
        elif key == 'q':
            cmd = make_twist(z_rot=turn)
        elif key == 'e':
            cmd = make_twist(z_rot=-turn)
        elif key == ' ':
            cmd = make_twist()
        elif key == 'x':
            pub.publish(make_twist())
            break
        else:
            cmd = make_twist()

        pub.publish(cmd)


if __name__ == '__main__':
    main()
```

### 3. Make it executable and build

```bash
chmod +x ~/catkin_ws/src/lab5_jetauto_control/scripts/teleop_key_control.py
cd ~/catkin_ws
catkin_make
source devel/setup.bash
```

### 4. Run the controller

```bash
rosrun lab5_jetauto_control teleop_key_control.py
```

### 5. Verify the command topic
In another terminal:

```bash
rostopic echo /jetauto_controller/cmd_vel
```

Press keys in the controller terminal and confirm that `Twist` messages are being published.


## Troubleshooting guide

| Problem | Likely cause | Check / fix |
|---|---|---|
| SSH does not connect | Not connected to JetAuto Wi‑Fi | Reconnect to robot SSID and retry `ssh jetauto@192.168.149.1` |
| `roslaunch` command not found | ROS environment not sourced | Run `source /opt/ros/melodic/setup.bash` |
| Robot does not move | Wrong topic or controller not running | Check `rostopic list` and `rostopic type /jetauto_controller/cmd_vel` |
| Robot moves unexpectedly | Competing app/controller | Stop app service and send zero Twist command |
| Python controller does not run | Missing executable permission | Run `chmod +x scripts/teleop_key_control.py` |
| Package not found | Workspace not built or sourced | Run `catkin_make` and `source devel/setup.bash` |
| Keyboard input behaves strangely | Terminal focus issue | Click inside the terminal running the keyboard node |


## Submission checklist
Submit a short lab report or notebook export containing the following evidence.

### Required evidence

- Photo of the robot before motion testing, showing safe placement.
- Screenshot or copied terminal output of successful SSH connection.
- Screenshot or copied terminal output of:

```bash
rostopic list
```

- Screenshot or copied terminal output of:

```bash
rostopic type /jetauto_controller/cmd_vel
```

- Evidence that a stop command was sent before and after movement.
- Evidence that the robot completed a small forward or rotation test.
- Evidence that your Lab 4 controller package, or the optional Lab 5 controller package, ran on the physical robot.

### Written answers

Answer the following questions:

1. Which command topic did the real JetAuto use?
2. What message type controlled the robot motion?
3. What safety check mattered most before movement?
4. How did the physical robot differ from Gazebo in speed, drift, delay, or response?
5. What would you improve in your controller before using it in a more complex navigation task?


## Suggested grading rubric

| Category | Weight | Expectations |
|---|---:|---|
| Safety and setup | 20% | Robot inspected, safe test area prepared, stop command used correctly |
| Connection and ROS verification | 20% | SSH/serial access demonstrated, ROS distribution and topics checked |
| Hardware controller launch | 20% | App service stopped, JetAuto controller launched, command topic verified |
| Movement test | 20% | Safe movement command executed and stopped; observations documented |
| Controller package transfer/use | 10% | Lab 4 or Lab 5 controller runs on physical robot |
| Reflection quality | 10% | Clear comparison between simulation and physical robot behavior |


# Appendix A — ROS2 version of this lab

This appendix is for students using a ROS2 environment instead of the ROS1 Melodic JetAuto image.

## A.1 Recommended ROS2 setup

For a modern student laptop or VM:

| Ubuntu version | Recommended ROS2 distribution |
|---|---|
| Ubuntu 24.04 | ROS2 Jazzy |
| Ubuntu 22.04 | ROS2 Humble |

For ROS2 Jazzy on Ubuntu 24.04:

```bash
sudo apt update
sudo apt install ros-jazzy-desktop python3-colcon-common-extensions
source /opt/ros/jazzy/setup.bash
```

For ROS2 Humble on Ubuntu 22.04:

```bash
sudo apt update
sudo apt install ros-humble-desktop python3-colcon-common-extensions
source /opt/ros/humble/setup.bash
```


## A.2 ROS1 to ROS2 command translation

| ROS1 command | ROS2 equivalent |
|---|---|
| `roscore` | Usually not needed; DDS discovery is built in |
| `rostopic list` | `ros2 topic list` |
| `rostopic echo /topic` | `ros2 topic echo /topic` |
| `rostopic type /topic` | `ros2 topic info /topic` |
| `rosnode list` | `ros2 node list` |
| `roslaunch package file.launch` | `ros2 launch package file.launch.py` |
| `rosrun package node` | `ros2 run package node` |
| `catkin_make` | `colcon build` |
| `source devel/setup.bash` | `source install/setup.bash` |


## A.3 ROS2 workspace setup

```bash
mkdir -p ~/ros2_ws/src
cd ~/ros2_ws
colcon build
source install/setup.bash
```

Add the source command to your shell configuration if desired:

```bash
echo "source /opt/ros/jazzy/setup.bash" >> ~/.bashrc
echo "source ~/ros2_ws/install/setup.bash" >> ~/.bashrc
```

Use `humble` instead of `jazzy` if your system uses ROS2 Humble.


## A.4 ROS2 movement command

In ROS2, the `Twist` message type is written with the `/msg/` namespace:

```bash
ros2 topic pub --once /cmd_vel geometry_msgs/msg/Twist \
"{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}"
```

Small forward test:

```bash
ros2 topic pub --once /cmd_vel geometry_msgs/msg/Twist \
"{linear: {x: 0.1, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}"
```

Immediately send the zero command again.

> If your ROS2 JetAuto controller uses a namespaced topic such as `/jetauto_controller/cmd_vel`, replace `/cmd_vel` with the correct topic.


## A.5 ROS2 keyboard controller option

Install the standard keyboard teleoperation package:

```bash
sudo apt update
sudo apt install ros-jazzy-teleop-twist-keyboard
```

For ROS2 Humble:

```bash
sudo apt install ros-humble-teleop-twist-keyboard
```

Run it on the default `/cmd_vel` topic:

```bash
ros2 run teleop_twist_keyboard teleop_twist_keyboard
```

Run it with a remapped JetAuto topic:

```bash
ros2 run teleop_twist_keyboard teleop_twist_keyboard --ros-args \
-r cmd_vel:=/jetauto_controller/cmd_vel
```

Use very small speed values when testing a real robot.


## A.6 Optional ROS2 Python controller package

Create a ROS2 Python package:

```bash
cd ~/ros2_ws/src
ros2 pkg create lab5_jetauto_control_ros2 --build-type ament_python --dependencies rclpy geometry_msgs
```

Create a simple publisher node in:

```text
~/ros2_ws/src/lab5_jetauto_control_ros2/lab5_jetauto_control_ros2/safe_forward.py
```

Example node:

```python
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist


class SafeForward(Node):
    def __init__(self):
        super().__init__('safe_forward')
        self.pub = self.create_publisher(Twist, '/cmd_vel', 10)
        self.timer = self.create_timer(0.5, self.publish_once)
        self.sent = False

    def publish_once(self):
        msg = Twist()
        if not self.sent:
            msg.linear.x = 0.1
            self.sent = True
        else:
            msg.linear.x = 0.0
        self.pub.publish(msg)
        self.get_logger().info(f'Published linear.x = {msg.linear.x}')


def main():
    rclpy.init()
    node = SafeForward()
    rclpy.spin_once(node, timeout_sec=1.5)
    node.destroy_node()
    rclpy.shutdown()


if __name__ == '__main__':
    main()
```

Then update `setup.py` with a console script entry point, rebuild, and run:

```bash
cd ~/ros2_ws
colcon build
source install/setup.bash
ros2 run lab5_jetauto_control_ros2 safe_forward
```


## A.7 ROS2 submission checklist

For ROS2 students, submit equivalent evidence:

- ROS2 distribution used: `jazzy` or `humble`.
- Output of:

```bash
ros2 topic list
```

- Output of:

```bash
ros2 topic info /cmd_vel
```

or the correct JetAuto command topic.

- Screenshot or copied terminal output showing safe `Twist` publishing.
- Evidence that the robot or simulator received the command.
- Short comparison explaining what changed between ROS1 and ROS2 commands.
